# VisionBridge — End-to-End Base Model Training (Colab)

Run every cell from top to bottom.

This notebook syncs the current `main` branch, downloads the real ISL-CSLTR sentence-level videos, builds a reproducible diverse real-data subset, extracts MediaPipe Holistic features (`pose=[T,132]`, `face=[T,1404]`), runs the 4-sample CTC overfit gate, trains the existing Pose+Face Transformer, evaluates a held-out validation split every epoch, prints decoded validation predictions every epoch, saves the best `base_model.pt` + vocabulary, validates the final checkpoint, and optionally pushes only the trained model artifacts to GitHub.

It does not redesign the model or modify backend source files.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_ROOT = Path('/content/VisionBridge')

if not (REPO_ROOT / 'README.md').exists():
    subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',str(REPO_ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_ROOT),'checkout','main'],check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'pull','--ff-only'],check=True)

BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0,str(BACKEND_ROOT))
os.chdir(REPO_ROOT)

print('Repo:',REPO_ROOT)
print('HEAD:',subprocess.check_output(['git','-C',str(REPO_ROOT),'rev-parse','--short','HEAD'],text=True).strip())
print('Branch:',subprocess.check_output(['git','-C',str(REPO_ROOT),'branch','--show-current'],text=True).strip())
print('SYNC: PASS')

In [ ]:
import torch
print('Python:',sys.version.split()[0])
print('PyTorch:',torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('GPU required for full training. In Colab choose Runtime > Change runtime type > T4 GPU.')
print('GPU:',torch.cuda.get_device_name(0))
torch.backends.cuda.matmul.allow_tf32 = True
print('GPU CHECK: PASS')

In [ ]:
import os, subprocess, shutil
from pathlib import Path
MP_ENV = Path('/content/visionbridge_mp312')
MP_PYTHON = MP_ENV / 'bin/python'
MPL_CONFIG = Path('/content/visionbridge_mplconfig')
MPL_CONFIG.mkdir(parents=True,exist_ok=True)
uv = shutil.which('uv')
if uv is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','uv'],check=True)
    uv = shutil.which('uv')
if uv is None:
    raise RuntimeError('uv is unavailable.')
if subprocess.run([uv,'python','find','3.12'],capture_output=True).returncode != 0:
    subprocess.run([uv,'python','install','3.12'],check=True)
if not MP_PYTHON.exists():
    subprocess.run([uv,'venv','--python','3.12',str(MP_ENV)],check=True)
mp_env = os.environ.copy()
mp_env['MPLBACKEND'] = 'Agg'
mp_env['MPLCONFIGDIR'] = str(MPL_CONFIG)
probe = subprocess.run([str(MP_PYTHON),'-c','import mediapipe; from mediapipe.python.solutions import holistic; print(mediapipe.__version__)'],text=True,capture_output=True,env=mp_env)
if probe.returncode != 0 or probe.stdout.strip() != '0.10.21':
    subprocess.run([uv,'pip','install','--python',str(MP_PYTHON),'mediapipe==0.10.21','numpy==1.26.4','opencv-python-headless','pandas','matplotlib'],check=True,env=mp_env)
probe = subprocess.run([str(MP_PYTHON),'-c',"import sys,os,mediapipe; from mediapipe.python.solutions import holistic; print(sys.version.split()[0]); print(mediapipe.__version__); print(os.environ.get('MPLBACKEND'))"],text=True,capture_output=True,env=mp_env)
print(probe.stdout)
if probe.returncode != 0:
    print(probe.stderr)
    raise RuntimeError('MediaPipe environment failed validation.')
print('MEDIAPIPE ENV: PASS')

In [ ]:
import glob, os, subprocess, sys
try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','kagglehub'],check=True)
    import kagglehub
dataset_path = kagglehub.dataset_download('drblack00/isl-csltr-indian-sign-language-dataset')
candidates = [p for p in glob.glob(os.path.join(dataset_path,'**','*Sentence_Level*'),recursive=True) if os.path.isdir(p) and 'Video' in os.path.basename(p)]
if len(candidates) != 1:
    raise RuntimeError(f'Expected one sentence-level video directory, found {len(candidates)}: {candidates}')
VIDEO_ROOT = candidates[0]
video_files = []
for ext in ('*.mp4','*.MP4','*.avi','*.AVI','*.mov','*.MOV'):
    video_files.extend(glob.glob(os.path.join(VIDEO_ROOT,'**',ext),recursive=True))
video_files = sorted(video_files)
if not video_files:
    raise RuntimeError(f'No sentence-level videos found under {VIDEO_ROOT}')
print('Dataset:',dataset_path)
print('Video root:',VIDEO_ROOT)
print('Total sentence videos:',len(video_files))

In [ ]:
import random
from pathlib import Path
SEED = 42
TARGET_SAMPLES = min(600,len(video_files))
MAX_PER_LABEL = 2
rng = random.Random(SEED)
by_label = {}
for vp in video_files:
    label = Path(vp).parent.name.replace('_',' ').strip()
    if label:
        by_label.setdefault(label,[]).append(vp)
labels = sorted(by_label)
rng.shuffle(labels)
selected = []
for label in labels:
    clips = list(by_label[label])
    rng.shuffle(clips)
    for clip in clips[:MAX_PER_LABEL]:
        selected.append((clip,label))
        if len(selected) >= TARGET_SAMPLES:
            break
    if len(selected) >= TARGET_SAMPLES:
        break
if len(selected) < 100:
    raise RuntimeError(f'Only {len(selected)} usable clips found; need at least 100.')
print('Selected clips:',len(selected))
print('Unique labels:',len({label for _,label in selected}))

In [ ]:
import csv, subprocess, textwrap
import numpy as np
DATA_DIR = REPO_ROOT / 'data' / 'processed' / 'isltranslate'
POSE_DIR = DATA_DIR / 'pose'
FACE_DIR = DATA_DIR / 'face'
CSV_PATH = DATA_DIR / 'ISLTranslate.csv'
DATA_DIR.mkdir(parents=True,exist_ok=True)
POSE_DIR.mkdir(parents=True,exist_ok=True)
FACE_DIR.mkdir(parents=True,exist_ok=True)
helper = REPO_ROOT / 'data' / 'model_check' / '_extract_train_one.py'
helper.parent.mkdir(parents=True,exist_ok=True)
helper_code = '''
import sys
from pathlib import Path
import numpy as np
from mediapipe.python.solutions import holistic
repo = Path(sys.argv[1]); video = sys.argv[2]; pose_out = Path(sys.argv[3]); face_out = Path(sys.argv[4])
sys.path.insert(0,str(repo / 'backend'))
from scripts.extract_keypoints import extract_clip_keypoints
with holistic.Holistic(static_image_mode=False,model_complexity=1) as solution:
    pose, face = extract_clip_keypoints(video,solution)
assert pose.ndim == 2 and pose.shape[1] == 132
assert face.ndim == 2 and face.shape[1] == 1404
assert pose.shape[0] == face.shape[0] and pose.shape[0] > 0
np.save(pose_out,pose); np.save(face_out,face)
print('frames=',pose.shape[0])
'''
helper.write_text(textwrap.dedent(helper_code),encoding='utf-8')
rows=[]; failed=[]
for idx,(video,text) in enumerate(selected,start=1):
    uid = Path(video).stem
    pose_path = POSE_DIR / f'{uid}.npy'
    face_path = FACE_DIR / f'{uid}.npy'
    if pose_path.exists() and face_path.exists():
        try:
            pose = np.load(pose_path,mmap_mode='r'); face = np.load(face_path,mmap_mode='r')
            if pose.ndim == 2 and pose.shape[1] == 132 and face.ndim == 2 and face.shape[1] == 1404 and pose.shape[0] == face.shape[0] and pose.shape[0] > 0:
                rows.append({'uid':uid,'text':text}); continue
        except Exception:
            pass
    result = subprocess.run([str(MP_PYTHON),str(helper),str(REPO_ROOT),video,str(pose_path),str(face_path)],text=True,capture_output=True,env=mp_env)
    if result.returncode != 0:
        failed.append((video,result.stderr[-500:])); continue
    rows.append({'uid':uid,'text':text})
    if idx % 25 == 0 or idx == len(selected):
        print(f'processed={idx}/{len(selected)} valid={len(rows)} failed={len(failed)}')
if len(rows) < 100:
    raise RuntimeError(f'Only {len(rows)} valid clips after extraction; need at least 100.')
with CSV_PATH.open('w',newline='',encoding='utf-8') as handle:
    writer = csv.DictWriter(handle,fieldnames=['uid','text']); writer.writeheader(); writer.writerows(rows)
valid_uids = {r['uid'] for r in rows}
for directory in (POSE_DIR,FACE_DIR):
    for path in directory.glob('*.npy'):
        if path.stem not in valid_uids:
            path.unlink()
print('Prepared real examples:',len(rows)); print('Failed extractions:',len(failed)); print('CSV:',CSV_PATH)

In [ ]:
from app.training.isltranslate import ISLTranslateKeypointDataset, SimpleCharTokenizer
tokenizer = SimpleCharTokenizer()
dataset = ISLTranslateKeypointDataset(DATA_DIR,tokenizer=tokenizer)
print('Examples:',len(dataset)); print('Vocabulary size:',tokenizer.vocab_size)
if len(dataset) < 100:
    raise RuntimeError('Too few processed examples for training.')
for i in range(min(5,len(dataset))):
    item = dataset[i]; pose, face = item['pose'], item['face']
    assert pose.ndim == 2 and pose.shape[1] == 132
    assert face.ndim == 2 and face.shape[1] == 1404
    assert pose.shape[0] == face.shape[0] > 0
    print(i,item['uid'],pose.shape,face.shape,item['text'])
print('DATASET CONTRACT: PASS')

In [ ]:
cmd = [sys.executable,'-m','app.training.overfit_sanity','--data-dir',str(DATA_DIR),'--samples','4','--steps','150']
sanity_env = os.environ.copy(); sanity_env['PYTHONPATH'] = str(BACKEND_ROOT)
result = subprocess.run(cmd,cwd=REPO_ROOT,env=sanity_env,text=True,capture_output=True)
print(result.stdout)
if result.stderr: print('STDERR:\n',result.stderr)
if result.returncode != 0:
    raise RuntimeError('OVERFIT SANITY FAILED. Do not run full training.')
print('OVERFIT SANITY GATE: PASS')

In [ ]:
import copy, random, time
import torch
from torch.utils.data import DataLoader, random_split
from app.models.base_model import VisionBridgeBaseModel
from app.services.inference_service import decode_logits
from app.training.isltranslate import collate_ctc_batch
TRAIN_EPOCHS = 15
BATCH_SIZE = 2
LR = 3e-4
WEIGHT_DECAY = 1e-2
MAX_GRAD_NORM = 1.0
VAL_FRACTION = 0.10
PATIENCE = 4
torch.manual_seed(SEED); random.seed(SEED)
val_size = max(1,int(len(dataset)*VAL_FRACTION)); train_size = len(dataset)-val_size
train_ds, val_ds = random_split(dataset,[train_size,val_size],generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate_ctc_batch,num_workers=0)
val_loader = DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_ctc_batch,num_workers=0)
device = torch.device('cuda')
model = VisionBridgeBaseModel(vocab_size=tokenizer.vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
loss_fn = torch.nn.CTCLoss(blank=0,zero_infinity=True)
output_path = REPO_ROOT / 'backend/app/models/weights/base_model.pt'
vocab_path = REPO_ROOT / 'backend/app/models/weights/base_model.vocab.json'
best_val_loss = float('inf'); best_state = None; patience_count = 0
def run_epoch(loader,training):
    model.train(training); total_loss=0.0; batches=0
    for batch in loader:
        pose=batch['pose'].to(device,non_blocking=True); face=batch['face'].to(device,non_blocking=True); labels=batch['labels'].to(device,non_blocking=True); input_lengths=batch['input_lengths'].to(device,non_blocking=True); label_lengths=batch['label_lengths'].to(device,non_blocking=True)
        with torch.set_grad_enabled(training):
            logits=model(pose,face,input_lengths)
            log_probs=torch.log_softmax(logits,dim=-1).transpose(0,1)
            loss=loss_fn(log_probs,labels,input_lengths,label_lengths)
            if training:
                optimizer.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),MAX_GRAD_NORM); optimizer.step()
        total_loss += float(loss.item()); batches += 1
    return total_loss/max(batches,1)
def preview_predictions(loader,limit=5):
    model.eval(); examples=[]
    with torch.inference_mode():
        for batch in loader:
            pose=batch['pose'].to(device); face=batch['face'].to(device); lengths=batch['input_lengths'].to(device); logits=model(pose,face,lengths)
            for i in range(logits.shape[0]):
                pred,conf=decode_logits(logits[i:i+1]); examples.append({'truth':batch['text'][i],'prediction':pred,'confidence':float(conf)})
                if len(examples) >= limit: return examples
    return examples
print('Train examples:',len(train_ds)); print('Validation examples:',len(val_ds)); print('Device:',torch.cuda.get_device_name(0))
for epoch in range(1,TRAIN_EPOCHS+1):
    start=time.time(); train_loss=run_epoch(train_loader,True); val_loss=run_epoch(val_loader,False); previews=preview_predictions(val_loader,5)
    print(f'\nEpoch {epoch}/{TRAIN_EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | time={time.time()-start:.1f}s')
    for i,item in enumerate(previews,1): print(f"  [{i}] truth={item['truth']!r} | pred={item['prediction']!r} | conf={item['confidence']:.3f}")
    if val_loss < best_val_loss:
        best_val_loss=val_loss; best_state=copy.deepcopy(model.state_dict()); torch.save(best_state,output_path); tokenizer.save(vocab_path); patience_count=0; print('  -> BEST CHECKPOINT SAVED')
    else:
        patience_count += 1
    if patience_count >= PATIENCE:
        print(f'Early stopping after {PATIENCE} non-improving epochs.'); break
if best_state is None:
    raise RuntimeError('No best checkpoint produced.')
model.load_state_dict(best_state); model.eval()
print('\nTRAINING COMPLETE'); print('Best validation loss:',best_val_loss); print('Checkpoint:',output_path); print('Vocabulary:',vocab_path)

In [ ]:
model.eval(); final_examples = preview_predictions(val_loader,limit=min(20,len(val_ds)))
print('FINAL VALIDATION')
nonblank = 0
for i,item in enumerate(final_examples,1):
    print(f"[{i}] truth={item['truth']!r}\n    pred ={item['prediction']!r} conf={item['confidence']:.3f}")
    if item['prediction'] != '(no sign detected)': nonblank += 1
print(f'Non-blank predictions: {nonblank}/{len(final_examples)}')
if nonblank == 0:
    raise RuntimeError('FINAL VALIDATION FAILED: checkpoint produces only blank predictions.')
print('FINAL VALIDATION: PASS')

In [ ]:
from app.models.base_model import load_frozen_base_model
assert output_path.exists() and vocab_path.exists()
reloaded = load_frozen_base_model(str(output_path),vocab_size=tokenizer.vocab_size).to(device).eval()
assert sum(p.numel() for p in reloaded.parameters() if p.requires_grad) == 0
size_mb = output_path.stat().st_size / (1024*1024)
print('Checkpoint size:',f'{size_mb:.2f} MB'); print('Vocabulary size:',tokenizer.vocab_size)
if size_mb >= 95:
    raise RuntimeError('Checkpoint is too close to GitHub 100 MB limit.')
print('CHECKPOINT LOAD: PASS')

In [ ]:
# Push ONLY the trained model artifacts.
# Add a Colab Secret named GITHUB_TOKEN before this cell.
import subprocess, os
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = os.environ.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('GITHUB_TOKEN is missing. Add a Colab Secret named GITHUB_TOKEN and rerun this final cell.')
remote = 'https://x-access-token:' + token + '@github.com/BharathWaj-K-R/VisionBridge.git'
subprocess.run(['git','-C',str(REPO_ROOT),'remote','set-url','origin',remote],check=True)
subprocess.run(['git','-C',str(REPO_ROOT),'add','backend/app/models/weights/base_model.pt','backend/app/models/weights/base_model.vocab.json'],check=True)
print(subprocess.check_output(['git','-C',str(REPO_ROOT),'status','--short'],text=True))
commit = subprocess.run(['git','-C',str(REPO_ROOT),'commit','-m','train: update validated VisionBridge base model'],text=True,capture_output=True)
print(commit.stdout)
if commit.returncode not in (0,1):
    print(commit.stderr); raise RuntimeError('Git commit failed.')
subprocess.run(['git','-C',str(REPO_ROOT),'push','origin','main'],check=True)
sha = subprocess.check_output(['git','-C',str(REPO_ROOT),'rev-parse','HEAD'],text=True).strip()
print('MODEL PUSH: PASS'); print('Commit:',sha); print('Only base_model.pt and base_model.vocab.json were staged by this cell.')